# BIST Enterprise RAG & Pipeline Jobs Architecture Guide

본 노트북은 `jobs/` 디렉토리에 정의된 **선언적 파이프라인 레시피(Declarative Pipeline Job Recipes)**와 **Kubernetes KEDA 비동기 워커 정책(Worker Policies)**의 아키텍처, 데이터 모델, 실행 흐름 및 세부 구현을 설명합니다.

---

## 1. Jobs 서브시스템 아키텍처 개요

BIST 시스템의 모든 백그라운드 및 파이프라인 작업은 `jobs/` 패키지 내에 **불변의 순수 선언적 데이터 계약(Pure Immutable Declarative Contracts)**으로 정의됩니다.

```
┌────────────────────────────────────────────────────────────────────────┐
│                      BaseJobDefinition                                 │
│  - job_id: str (고유 식별자)                                           │
│  - name: str (사람이 읽을 수 있는 이름)                                │
│  - description: str (작업 상세 설명)                                   │
│  - queue_name: str (PostgreSQL 큐 테이블의 대상 큐 이름)               │
│  - version: str (레시피 버전)                                          │
└───────────────────┬────────────────────────────────┬───────────────────┘
                    │                                │
                    ▼                                ▼
   ┌────────────────────────────────┐ ┌────────────────────────────────┐
   │       DagJobDefinition         │ │      WorkerJobDefinition       │
   │ - nodes: Tuple[JobNode, ...]   │ │ - worker_entrypoint: str       │
   │ - edges: Tuple[JobEdge, ...]   │ │ - kubernetes: WorkerPolicy     │
   │ (공용 Workflow Worker가 실행)  │ │ (독립 Pod/Leased Worker 실행)  │
   └────────────────────────────────┘ └────────────────────────────────┘
```

### 주요 설계 원칙
1. **Zero Side-Effects**: Job 정의 객체는 DB 연결이나 네트워크 I/O 없이 순수 데이터(DataClass)로만 구성됩니다.
2. **Dual Execution Model**:
   - **DAG Jobs**: 모듈들을 토폴로지 순서로 연결한 DAG로, 공용 `WorkflowWorker`(`backend.engine.worker.main`)가 병렬/순차 실행합니다.
   - **Worker Jobs**: 고유한 엔트리포인트를 가지며, KEDA(Kubernetes Event-driven Autoscaling)가 PostgreSQL 큐 상태에 따라 전용 워커 Pod를 오토스케일링합니다.

In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 디렉토리 설정
# Find repository root by walking up from cwd to find jobs/__init__.py
current = Path.cwd()
PROJECT_ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "jobs" / "__init__.py").exists():
        PROJECT_ROOT = parent
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find repository root containing jobs/__init__.py")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jobs import (
    ALL_JOBS,
    JOB_REGISTRY,
    get_job_definition,
    DagJobDefinition,
    WorkerJobDefinition,
    RAG_QUERY_JOB,
    EXCEL_INGESTION_JOB,
    BI_METRIC_EXTRACTION_JOB,
    BI_MATERIALIZATION_JOB,
    BI_QUESTION_JOB,
    BENCHMARK_JOB,
)
from jobs.kubernetes import kubernetes_worker_specs

print(f"✅ Successfully loaded {len(ALL_JOBS)} jobs from jobs package!")
for job in ALL_JOBS:
    job_type = "DAG Module Pipeline" if isinstance(job, DagJobDefinition) else "Dedicated Worker Job"
    print(f"  - [{job.job_id}] {job.name} (Type: {job_type}, Queue: '{job.queue_name}', Version: {job.version})")

## 2. DAG Module Pipeline Jobs 심층 분석

`DagJobDefinition`은 `modules/` 디렉토리에 구현된 독립적인 RAG 파이프라인 모듈들을 노드(`JobNode`)와 데이터 바인딩 엣지(`JobEdge`)로 연결한 실행 청사진입니다.

---

### 2.1 `RAG_QUERY_JOB` (하이브리드 재무 질의응답 파이프라인)
- **Job ID**: `rag_query`
- **Target Queue**: `workflow-core`
- **Architecture**:
  1. `query` (`query_input`): 사용자 자연어 질문 수신 및 컨텍스트 초기화
  2. `decompose` (`decomposer`): 복합 질문을 연도·계정 항목 단위 원자적 서브쿼리로 분해 (LLM)
  3. `data-scope` (`pgvector_data_scope`): 사용 가능한 기업/인덱스/시트 카탈로그 조회
  4. `route` (`llm_query_router`): 서브쿼리를 최적의 재무제표 시트로 매핑
  5. `embed-query` (`embedder`): 라우팅된 서브쿼리 텍스트를 고차원 벡터로 인코딩
  6. `dense` (`pgvector_retriever`): pgvector HNSW 인덱스 기반 Top-K 코사인 유사도 검색
  7. `keyword` (`postgres_native_keyword_retriever`): PostgreSQL Full-Text Search (BM25 형태 키워드 검색)
  8. `fuse` (`rrf_fusion`): Dense 점수와 Keyword 순위를 RRF(Reciprocal Rank Fusion) 공식으로 결합
  9. `expand-context` (`pg_context_expander`): 검색된 핵심 셀 주변 행/열 헤더 및 인접 수치 셀 컨텍스트 확장
  10. `read` (`reader`): 확장된 셀 테이블과 LangChain 수식 계산 툴을 활용하여 근거 기반 최종 답변 생성

In [ ]:
def inspect_dag_job(job: DagJobDefinition):
    print(f"=======================================================================")
    print(f"📌 [DAG Job Blueprint] {job.name} (ID: {job.job_id})")
    print(f"=======================================================================")
    print(f"📝 Description: {job.description}")
    print(f"📬 Target Queue: {job.queue_name}")
    print(f"🔢 Nodes count: {len(job.nodes)} | Edges count: {len(job.edges)}\n")
    
    print("--- [Nodes (Modules)] ---")
    for i, node in enumerate(job.nodes, 1):
        print(f"  {i:02d}. Node ID: '{node.node_id}' -> Module Type: '{node.module_type}'")
        
    print("\n--- [Data Flow Topology (Edges)] ---")
    for i, edge in enumerate(job.edges, 1):
        print(f"  {i:02d}. [{edge.source}.{edge.source_output}] ===> [{edge.target}.{edge.target_input}] (Edge: {edge.edge_id})")
    print("\n")

inspect_dag_job(RAG_QUERY_JOB)

### 2.2 `EXCEL_INGESTION_JOB` (엑셀 비정형 파싱 & pgvector 인덱싱)
- **Job ID**: `excel_ingestion`
- **Target Queue**: `workflow-core`
- **Architecture**:
  1. `source` (`processed_file_selector`): 업로드된 원본 엑셀 파일 메타데이터 검증
  2. `structure` (`luna_vlm_structure_detector`): Luna Vision-Language 모델을 통해 복합 헤더, 병합 셀, 표 영역 감지
  3. `serialize` (`cell_text_serializer`): 기업명, 시트명, 계층형 행/열 헤더를 셀 좌표와 함께 자연어 텍스트로 직렬화
  4. `embed` (`cell_text_embedder`): 직렬화된 셀 텍스트를 배치 임베딩 벡터로 변환
  5. `write-index` (`pgvector_index_writer`): `langchain_pg_embedding` 테이블에 PostgreSQL Binary COPY로 대량 적재
  6. `persist-sheets` (`sheet_metadata_persistence`): 시트별 컬렉션 및 헤더 구조 DB 저장
  7. `persist-company` (`company_entity_extractor`): 기업명 식별 및 인덱스 바인딩 영속화

In [ ]:
inspect_dag_job(EXCEL_INGESTION_JOB)

### 2.3 `BI_METRIC_EXTRACTION_JOB` (BI 재무 메트릭 추출 파이프라인)
- **Job ID**: `bi_metric_extraction`
- **Target Queue**: `workflow-core`
- **Role**: BI 대시보드 구축을 위해 18개 핵심 재무 지표(매출, 영업이익, FCF, 부채비율 등)와 분기/연도별 시계열 값을 정밀 추출하는 전용 DAG 파이프라인입니다.

In [ ]:
inspect_dag_job(BI_METRIC_EXTRACTION_JOB)

## 3. Dedicated Worker Jobs & Kubernetes KEDA Autoscaling

`WorkerJobDefinition`은 고유한 Python 엔트리포인트를 가지고 백그라운드에서 동작하는 작업 단위입니다. KEDA ScaledJob과 연동되어 PostgreSQL 큐에 대기 중인 레코드가 발생하면 자동으로 Pod를 스폰합니다.

---

In [ ]:
def inspect_worker_job(job: WorkerJobDefinition):
    print(f"=======================================================================")
    print(f"⚙️ [Dedicated Worker Job] {job.name} (ID: {job.job_id})")
    print(f"=======================================================================")
    print(f"📝 Description: {job.description}")
    print(f"📬 Queue Name: {job.queue_name}")
    print(f"🚀 Python Entrypoint: {job.worker_entrypoint}")
    print(f"📦 Worker Module: {job.worker_module}")
    print(f"\n--- [Kubernetes Worker Policy] ---")
    print(f"  - Deployment Name: {job.kubernetes.deployment_name}")
    print(f"  - Active Deadline: {job.kubernetes.active_deadline_seconds} sec ({job.kubernetes.active_deadline_seconds / 60:.1f} min)")
    print(f"  - Mount Data Volume: {job.kubernetes.mount_data_volume}")
    print(f"  - KEDA Scaling Trigger SQL Query:")
    for line in job.kubernetes.pending_query.strip().split("\n"):
        print(f"      {line}")
    print("\n")

inspect_worker_job(BI_MATERIALIZATION_JOB)
inspect_worker_job(BI_QUESTION_JOB)
inspect_worker_job(BENCHMARK_JOB)

## 4. Kubernetes Worker Spec 생성 및 KEDA ScaledJob 배포 계약

`jobs.kubernetes.kubernetes_worker_specs(ALL_JOBS)` 함수는 모든 등록된 Job을 분석하여:
1. 모든 `DagJobDefinition`의 큐(`workflow-core` 등)를 처리하는 공용 워커 스펙
2. 각 `WorkerJobDefinition`별 전용 워커 스펙

을 생성하고, KEDA `ScaledJob` 트리거 쿼리와 Pod 타임아웃 메타데이터를 결정론적으로 출력합니다.

In [ ]:
specs = kubernetes_worker_specs(ALL_JOBS)

print(f"📊 Total Kubernetes Worker Specifications: {len(specs)}\n")
for i, spec in enumerate(specs, 1):
    print(f"{i}. Deployment: '{spec.deployment_name}' (App: '{spec.app_name}')")
    print(f"   - Target Queue: '{spec.queue_name}'")
    print(f"   - Worker Module: '{spec.worker_module}'")
    print(f"   - Command Arguments: {spec.arguments}")
    print(f"   - Mount Data Volume: {spec.mount_data_volume}")
    print(f"   - Active Deadline: {spec.active_deadline_seconds}s")
    print(f"   - Trigger SQL Query Preview: {spec.pending_query.splitlines()[0]} ...")
    print("-" * 70)

## 5. Summary & Key Takeaways

| 구분 | Job ID | 유형 | 큐 (Queue) | 역할 | 워커 구현체 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **RAG Query** | `rag_query` | DAG | `workflow-core` | 하이브리드 RAG 자연어 질의응답 파이프라인 | `backend.engine.worker.main` |
| **Excel Ingestion** | `excel_ingestion` | DAG | `workflow-core` | 비정형 엑셀 구조화 및 pgvector 인덱싱 | `backend.engine.worker.main` |
| **BI Metric Extraction** | `bi_metric_extraction` | DAG | `workflow-core` | 18개 재무 지표 정밀 RAG 추출 | `backend.engine.worker.main` |
| **BI Materialization** | `bi_materialization` | Worker | `bi-materialization` | 기업별 대시보드 스냅샷 생성 코디네이터 | `backend.features.bi.materialization_worker_main` |
| **BI Question Batch** | `bi_question` | Worker | `bi-question` | 16개 배치 & 멀티스레드 병렬 질문 처리 | `backend.features.bi.question_worker_main` |
| **Benchmark** | `benchmark` | Worker | `benchmark` | RAG 정확도/지연/비용 비교 평가 및 채점 | `backend.features.benchmark.worker_main` |

---